<a href="https://colab.research.google.com/github/Asheesh1272/Asheesh127/blob/main/Fourier_Transform4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%matplotlib inline


In [ ]:
import numpy as np
import math
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import IPython.display as display
import os
from skimage.io import imread
from collections import namedtuple

In [ ]:
input_folder = "/content/input/images"

In [ ]:
def animate_sine():
    min_x = 0
    max_spiral_x = 1

    time_period = 1 #seconds
    Fs = 200.0 #sampling rate
    Ts = time_period/Fs # sampling interval

    cycle = 2*np.pi
    frequency = 4

    def sine_wave(n):
        return np.sin(n*cycle*frequency)

    def g(n):
        return (np.exp(cycle*1j*n)) * sine_wave(n)

    x = np.arange(min_x, max_spiral_x+Ts, Ts)

    fig, (ax_spiral, ax_sine) = plt.subplots(1, 2)
    fig.suptitle("The Sine wave with frequency %d Hz wrapped in a spiral" % frequency)

    p_init = g(x)
    line, = ax_spiral.plot([], [], 'g-', animated=True)
    point_of_spiral, = ax_spiral.plot(p_init.real, p_init.imag, "ro", animated=True)

    ax_spiral.set_xlabel("Real")
    ax_spiral.set_ylabel("Imaginary")
    ax_spiral.plot(np.real(g(x)), np.imag(g(x)))

    ax_sine.plot(x, sine_wave(x))
    point_of_sine, = ax_sine.plot(0, 0, "ro", animated=True)

    def update(x):
        p = g(x)
        line_x = np.linspace(0, p.real)
        # Ensure line_y is always a sequence (numpy array)
        if p.real != 0:
            line_y = (p.imag/p.real)*line_x
        else:
            line_y = np.zeros_like(line_x)
        line.set_data(line_x, line_y)
        point_of_spiral.set_data([p.real], [p.imag]) # Fixed: Pass as sequences
        point_of_sine.set_data([x], [sine_wave(x)]) # Fixed: Pass as sequences
        return point_of_spiral, line, point_of_sine

    anim = animation.FuncAnimation(fig,
                                   update,
                                   frames=np.linspace(min_x, max_spiral_x, 80, endpoint=False),
                                   interval=60,
                                   blit=True)
    return anim

anim = animate_sine()
display.HTML(anim.to_jshtml())

In [ ]:
def dft1D_base(x, sign, factor):
    """
    Discrete Fourier Transform 1D
    """
    N = len(x)
    X = np.array([])
    for k in range(0, N):
        Xk = complex(0, 0) # Changed np.complex to complex
        for n in range(0, N):
            exp = np.exp(sign*2*math.pi*k*n*1j/N)
            Xk += complex(x[n]) * exp
        X = np.append(X, [factor * Xk])
    return X

def dft1D(x):
    return dft1D_base(x, -1, 1)

def idft1D(X):
    return dft1D_base(X, 1, 1/len(X))

In [ ]:
time_period = 1  # 1 second
Fs = 100.0
Ts = time_period/Fs

t = np.arange(0, time_period, Ts)  # time vector in seconds
N = t.size
f = np.linspace(0, 1/Ts, N)

cycle = 2*np.pi
frequency1 = 1 * cycle
frequency2 = 3 * cycle

y = np.sin(t*frequency1) + np.sin(t*frequency2)
dft = dft1D(y)
idft = idft1D(dft)

plt.plot(t, y)
plt.title("Signal in the time domain")
plt.xlabel("Time [s]")
plt.ylabel("Amplitude")
plt.show()


In [ ]:
plt.bar(f[:N//2], np.abs(dft)[:N//2]*2/N, width=0.3)
plt.title("Signal in the frequency domain")
plt.xticks(range(0, 51, 2))
plt.xlabel("Frequency [Hertz]")
plt.ylabel("Amplitude")
plt.show()

In [ ]:
def extract_sines(possible_frequencies, dft_frequencies):
    """
    Extract sines from the frequency signal
    """
    functions = np.array([])
    for (frequency, dft) in zip(possible_frequencies, dft_frequencies):
        amplitude = math.sqrt(dft.real**2 + dft.imag**2)
        if(amplitude > 0.000001):
            # break the direct tie of the variables inside the main lambda
            # wrap them in another lambda called in the loop
            def sine(a, f): return (lambda t: (
                (1/a) * math.sin(f*2*math.pi*t)))
            wrapped_sine = np.vectorize((sine)(amplitude, frequency))
            functions = np.append(functions, [wrapped_sine])
    return functions

for sine in extract_sines(f[:N//2], dft[:N//2] * 2/N):
    plt.plot(t, sine(t))
plt.title("The extracted Signals in the time domain")
plt.xlabel("Time [s]")
plt.ylabel("Amplitude")
plt.show()

In [ ]:
plt.plot(t, np.real(idft))
plt.title("The signal retrieved by the inverse DFT")
plt.xlabel("Time [s]")
plt.ylabel("Amplitude")
plt.show()

In [ ]:
N = 32
x = np.random.random(N)
dft = dft1D(x)
fft = np.fft.fft(x)
ifft = np.fft.ifft(fft)
print("DFT 1D has similar results as Numpy FFT: ", np.allclose(dft, fft))
print("Inverse DFT 1D has similar results as Numpy IFFT: ", np.allclose(idft1D(dft), ifft))

In [ ]:
x = np.random.random(100)
%timeit dft1D(x)
%timeit np.fft.fft(x)

In [ ]:
def fft1D_base(x, sign, factor, ft1D):
    # the input must be a power of 2
    # in order to work with arbitrary dimensions
    # to implement padding
    N = len(x)
    if(N <= 1):
        return x
    M = (int)(N/2)
    even = np.array(ft1D(x[0::2]))
    odd = np.array(ft1D(x[1::2]))

    X = np.zeros(N, dtype=complex)
    for k in range(0, M):
        exp = np.exp(sign*2*math.pi*k*1j/N)
        oddTerm = exp*odd[k]
        X[k] = factor*(even[k] + oddTerm)
        X[k+M] = factor*(even[k] - oddTerm)
    return X

def fft1D(x):
    return fft1D_base(x, -1, 1, fft1D)

def ifft1D(freqs):
    return fft1D_base(freqs, 1, 0.5, ifft1D)

In [ ]:
N = 32
x = np.random.random(N)
fft = np.fft.fft(x)
custom_fft = fft1D(x)
ifft = np.fft.ifft(fft)
print("FFT 1D matches Numpy FFT: ", np.allclose(custom_fft, fft))
print("Inverse FFT 1D matches Numpy FFT: ", np.allclose(ifft1D(custom_fft), ifft))

In [ ]:
def ft2D(image, ft):
    M, N = image.shape
    transformed = np.empty((M, N), dtype=complex)

    # DFT 1D on rows
    for m in range(0, M):
        transformed[m, :] = ft(image[m, :])

    # DFT 1D on columns
    for n in range(0, N):
        transformed[:, n] = ft(transformed[:, n])

    return transformed

def fft2D(image):
    return ft2D(image, fft1D)

def ifft2D(image):
    return ft2D(image, ifft1D)

In [ ]:
def logabs(m):
    return np.log(1+np.abs(m))

def shiftlog(m):
    return np.fft.fftshift(logabs(m))

In [ ]:
def test_fft2D_with_small_image_pulses(size1, size2, title):
    columns, rows = 128, 128
    image = np.zeros((columns, rows))
    square1 = (10, columns/2, size1) # initial point + size
    square2 = (118, columns/2, size2)
    def in_square(r, c, square):
        (square_x, square_y, square_size) = square
        return square_x<=r and r<square_x+square_size and \
            square_y<=c and c<square_y+square_size

    for c in range(0, columns+1):
        for r in range(0, rows+1):
            if(in_square(r, c, square1) or in_square(r, c, square2)):
                image[c, r] = 1
    fft = fft2D(image)

    cm = "gray"
    _, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 10))
    ax1.imshow(image, cmap=cm)
    ax1.set_title(title)
    ax2.imshow(np.fft.fftshift(np.abs(fft)), cmap=cm, extent=[-columns/2, columns/2, -rows/2, rows/2])
    ax2.set_title("Frequency spectrum")
    plt.setp([ax1], xticks=[], yticks=[])

test_fft2D_with_small_image_pulses(2, 2, "Two dots")
test_fft2D_with_small_image_pulses(6, 6, "Two squares")

In [ ]:
def test_fft2D_with_sinusoids(frequency1, frequency2, title=""):
    columns, rows = 128, 128
    x = np.linspace(0, 1, rows)
    y = np.linspace(0, 1, rows)
    X = np.repeat(x[np.newaxis, :], rows, axis=0)
    Y = np.repeat(y[:, np.newaxis], columns, axis=1)
    sinusoid = np.sin(frequency1*2*np.pi*X) + np.sin(frequency2*2*np.pi*Y)

    fft = fft2D(sinusoid)

    cm = "gray"
    _, (ax_orig, ax_freq) = plt.subplots(1, 2, figsize=(10, 10))
    ax_orig.imshow(sinusoid, cmap=cm)
    ax_orig.set_title("Sinusoid: %s" % title)

    square = 150
    # zoom-in
    ax_freq.imshow(np.fft.fftshift(np.abs(fft))[-square:square, -square:square],
                   cmap=cm, extent=[-square/2, square/2, -square/2, square/2])
    ax_freq.set_title("Zoomed-in frequency spectrum FFT")

    plt.setp([ax_orig], xticks=[], yticks=[])

test_fft2D_with_sinusoids(4, 0, r"$ sin(4(2\pi)x) $")
test_fft2D_with_sinusoids(11, 0, r"$ sin(11(2\pi)x) $")
test_fft2D_with_sinusoids(3, 11, r"$ sin(3(2\pi)x) + sin(11(2\pi)y) $")

In [ ]:
Point = namedtuple('Point', 'x y')

def distance(p1, p2):
    return math.sqrt((p1.x-p2.x)**2 + (p1.y-p2.y)**2)

def image_filter(image, condition, default_value, filter_value):
    (columns, rows) = image.shape
    center = Point(rows/2, columns/2)

    base = []
    if default_value == 1:
        base = np.ones((columns, rows))
    else:
        base = np.zeros((columns, rows))

    for r in range(0, rows):
        for c in range(0, columns):
            if(condition(distance(Point(r, c), center))):
                base[c, r] = filter_value
    return base

def high_pass_filter(image, threshold=50):
    def hp(dist):
        return dist < threshold
    return image_filter(image, hp, 1, 0)

def low_pass_filter(image, threshold=50):
    def lp(dist):
        return dist < threshold
    return image_filter(image, lp, 0, 1)

In [ ]:
!npm install cameraman

In [ ]:
original_image = imread(os.path.join(input_folder, "/content/7B9ME.png"))
# Convert the image to grayscale if it's a color image
if original_image.ndim == 3:
    original_image = np.mean(original_image, axis=2)

fft = fft2D(original_image)
shifted_fft = np.fft.fftshift(fft)
hp_fft = shifted_fft * high_pass_filter(shifted_fft.copy())
lp_fft = shifted_fft * low_pass_filter(shifted_fft.copy())
hp = ifft2D(hp_fft)
lp = ifft2D(lp_fft)

(rows, columns) = original_image.shape
rows_half = (int)(rows/2)
columns_half = (int)(columns/2)
cm = "gray"
_, ((ax_orig, ax_hp, ax_lp),
    (ax_orig_freqs, ax_hp_freqs, ax_lp_freqs)) = plt.subplots(2, 3, figsize=(10, 10))
ax_orig.imshow(original_image, cmap=cm)
ax_orig.set_title("Original Image")
ax_hp.imshow(np.abs(hp), cmap=cm)
ax_hp.set_title("High Pass Filter")
ax_lp.imshow(np.abs(lp), cmap=cm)
ax_lp.set_title("Low Pass Filter")

ax_orig_freqs.imshow(shiftlog(fft), cmap=cm,
    extent=[-columns_half, columns_half, -rows_half, rows_half])
ax_orig_freqs.set_title("Frequency spectrum FFT")
ax_hp_freqs.imshow(logabs(hp_fft), cmap=cm, \
    extent=[-columns_half, columns_half, -rows_half, rows_half])
ax_hp_freqs.set_title("Frequency High Pass Filter")
ax_lp_freqs.imshow(logabs(lp_fft), cmap=cm, \
    extent=[-columns_half, columns_half, -rows_half, rows_half])
ax_lp_freqs.set_title("Frequency Low Pass Filter")

plt.setp([ax_orig, ax_hp, ax_lp], xticks=[], yticks=[])
plt.show()

In [ ]:
def custom_noise_filter(image,
                 notch_size,
                 threshold_for_unusual_peaks,
                 threshold_for_lp=50):

    columns, rows = image.shape
    base = np.ones((columns, rows))
    center = Point(rows/2, columns/2)
    notch_half = notch_size//2
    for r in range(0, rows):
        for c in range(0, columns):
            if(distance(Point(r, c), center) >= threshold_for_lp and
                    np.abs(image[c, r]) > threshold_for_unusual_peaks):
                for n in range(max(r-notch_half, 0), min(r+notch_half, rows)):
                    for m in range(max(c-notch_half,0), min(c+notch_half, columns)):
                        base[m, n] = 0
    return base

In [ ]:
def test_fft2D_with_noise_removal():
    original_image = imread(os.path.join(input_folder, "/content/FT-Moon land.jpg"))
    # Convert the image to grayscale if it's a color image
    if original_image.ndim == 3:
        original_image = np.mean(original_image, axis=2)

    fft = fft2D(original_image)
    shifted_fft = np.fft.fftshift(fft)
    noise_filter = custom_noise_filter(shifted_fft.copy(),
                                notch_size=9,
                                threshold_for_unusual_peaks=300000)

    filtered_fft = shifted_fft * noise_filter

    ifft = ifft2D(filtered_fft)

    (rows, columns) = filtered_fft.shape
    rows_half = (int)(rows/2)
    columns_half = (int)(columns/2)

    cm = "gray"
    _, ((ax_orig, ax_filtered), (ax_orig_freqs, ax_filtered_freqs)
        ) = plt.subplots(2, 2, figsize=(10, 10))
    ax_orig.imshow(original_image, cmap=cm)
    ax_orig.set_title("Original Image")
    ax_filtered.imshow(np.abs(ifft), cmap=cm)
    ax_filtered.set_title("Removed noise")

    ax_orig_freqs.imshow(shiftlog(fft), cmap=cm,
        extent=[-rows_half, rows_half, -columns_half, columns_half])
    ax_orig_freqs.set_title("Frequency spectrum FFT")
    ax_filtered_freqs.imshow(logabs(filtered_fft), cmap=cm,
        extent=[-rows_half, rows_half, -columns_half, columns_half])
    ax_filtered_freqs.set_title("Noise Removal Filter")

    plt.setp([ax_orig, ax_filtered], xticks=[], yticks=[])
test_fft2D_with_noise_removal()